# C4. Coarse conditioning noise

Does the fine level do better when trained not to trust its coarse input exactly?

See `docs/plans/experiments-plan.md`, entry C4. Results are read from `artifacts/results/C4/`. Nothing here trains or samples.

## Run

Run these from the repo root, in order. `uv run erosion-sim list` shows what is already trained.

**1. Data.** Skip any that already exist.

```sh
sbatch slurm/datagen.sbatch main_512
```

**2. Train.** One command per model; each resumes if stopped.

```sh
uv run erosion-sim train C4 fine_0.05
uv run erosion-sim train C4 fine_0.1
uv run erosion-sim train C4 fine_0.2
uv run erosion-sim train C4 fine_0.4
```

- `coarse` is `C1/coarse`, trained in C1. If it is not trained yet, `uv run erosion-sim train C4 coarse` trains it there.
- `fine_0` is `C1/fine`, trained in C1. If it is not trained yet, `uv run erosion-sim train C4 fine_0` trains it there.

**3. Evaluate.**

```sh
uv run erosion-sim evaluate C4
```

On the cluster, `uv run erosion-sim submit C4` does steps 2 and 3 as Slurm jobs.

Then run the cells below.

In [ ]:
import json

from erosion_sim.erosion import plots
from erosion_sim.experiments import registry
from erosion_sim.experiments.results import read_results
from erosion_sim.experiments.runner import model_folder
from erosion_sim.training.train import plot_history

NAME = "C4"
exp = registry.get(NAME)
results = read_results(NAME)
print(exp.question)
{"models": {label: m["folder"] for label, m in results.manifest["models"].items()},
 **{k: results.manifest[k] for k in ("code", "time", "slurm_job")}}

## Training

In [ ]:
for label in exp.models:
    folder = model_folder(NAME, label)[0]
    print(label, folder)
    plot_history(json.loads((folder / "history.json").read_text()))

## Worlds, levels alone, and baselines, on the runs' interiors

In [ ]:
plots.table(results, plots.TABLE + ('chunk_edge_rmse', 'seconds_per_query'))

In [ ]:
if "dT" in results.scalars["by"]:
    plots.by_table(results, "dT", "skill", kinds=("world", "baseline"))

## Seams

How much neighboring windows disagree before blending, per level and phase, as a share of the change.

In [ ]:
for world, levels in results.scalars["seams"].items():
    for level, phases in levels.items():
        print(world, level, [f"{p['relative']:.1%}" for p in phases])

## Examples

In [ ]:
plots.plot_examples(results, kinds=("world", "baseline"))

## Conclusion

_Written after reading the results above._